In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Set your directory paths
base_dir = '/content/drive/MyDrive/SemEval2026/'
# data_dir = base_dir + 'data/subtask2/'  # NOTE: subtask2, not subtask3!

# Install required packages
!pip install -q transformers datasets
!pip install -q accelerate

import pandas as pd
import numpy as np
import random
import math
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)

import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [3]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # Fallback to CPU for unsupported ops
torch.set_default_device("cpu")  # Force CPU as default device

# Verify CPU is being used
print(f"PyTorch using device: {torch.device('cpu')}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")

PyTorch using device: cpu
CUDA available: True
MPS available: False


In [4]:
label_cols = [
    "political",
    "racial/ethnic",
    "religious",
    "gender/sexual",
    "other"
]

NUM_LABELS = len(label_cols)
print(f"Number of labels: {NUM_LABELS}")
print(f"Labels: {label_cols}")

languages = ["eng","spa","arb","deu","zho"]

train_dfs = {}
dev_dfs = {}

print("\nLoading data...")
for lang in languages:
    train_path = base_dir + f"train/{lang}.csv"
    dev_path = base_dir + f"dev/{lang}.csv"

    train_dfs[lang] = pd.read_csv(train_path)
    train_dfs[lang]["language"] = lang
    dev_dfs[lang] = pd.read_csv(dev_path)
    dev_dfs[lang]["language"] = lang

    print(f"{lang}: {len(train_dfs[lang])} train, {len(dev_dfs[lang])} dev")

# Combine all languages
train_full = pd.concat([train_dfs[lang] for lang in languages], ignore_index=True)
dev_full = pd.concat([dev_dfs[lang] for lang in languages], ignore_index=True)

print(f"\nTotal train samples: {len(train_full)}")
print(f"Total dev samples: {len(dev_full)}")

Number of labels: 5
Labels: ['political', 'racial/ethnic', 'religious', 'gender/sexual', 'other']

Loading data...
eng: 2676 train, 133 dev
spa: 3305 train, 165 dev
arb: 3380 train, 169 dev
deu: 3180 train, 159 dev
zho: 4280 train, 214 dev

Total train samples: 16821
Total dev samples: 840


In [5]:

print("\n" + "="*60)
print("DATA EXPLORATION")
print("="*60)

# Check first few rows
print("\nSample data:")
print(train_full.head())

# Check label distribution
print("\nLabel distribution in training data:")
for col in label_cols:
    if col in train_full.columns:
        count = train_full[col].sum()
        pct = (count / len(train_full)) * 100
        print(f"{col}: {count} ({pct:.1f}%)")
    else:
        print(f"WARNING: Column '{col}' not found in data!")
        print(f"Available columns: {train_full.columns.tolist()}")

# Check multi-label statistics
print("\nMulti-label statistics:")
train_full['num_labels'] = train_full[label_cols].sum(axis=1)
print(train_full['num_labels'].value_counts().sort_index())

# Language distribution
print("\nLanguage distribution:")
print(train_full['language'].value_counts())


DATA EXPLORATION

Sample data:
                                    id  \
0  en_973938b90b0ff5d87d35a582f83f5c89   
1  en_07dfd4600426caca6e2c5883fcbea9ea   
2  en_f14519ff2302b6cd47712073f13bc461   
3  en_e48b7e7542faafa544ac57b64bc80daf   
4  en_7c581fb77bce8033aeba3d6dbd6273eb   

                                                text  political  \
0           is defending imperialism in the dnd chat          0   
1  Still playing with this. I am now following Ra...          0   
2  .senate.gov Theres 3 groups out there Republic...          0   
3  "ABC MD, David Anderson, said the additional f...          0   
4  "bad people" I have some conservative values s...          0   

   racial/ethnic  religious  gender/sexual  other language  
0              0          0              0      0      eng  
1              0          0              0      0      eng  
2              0          0              0      0      eng  
3              0          0              0      0      eng  
4      

In [6]:
train, validation = train_test_split(
    train_full,
    test_size=0.2,
    random_state=42,
    stratify=train_full["language"]
)

print(f"\nSplit sizes:")
print(f"Train: {len(train)}")
print(f"Validation: {len(validation)}")


Split sizes:
Train: 13456
Validation: 3365


In [7]:
GLOBAL_SEED = 42

def seed_everything(seed: int):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)

seed_everything(GLOBAL_SEED)
print(f"\nRandom seed set to: {GLOBAL_SEED}")


Random seed set to: 42


In [8]:
class PolarizationTypeDataset(torch.utils.data.Dataset):
    """
    Dataset for multi-label polarization type classification
    """
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors='pt'
        )

        item = {key: encoding[key].squeeze() for key in encoding.keys()}
        # Multi-label requires float labels
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

In [9]:
def compute_metrics_multilabel(p):
    """
    Compute macro F1 and accuracy for multi-label classification
    """
    logits = torch.tensor(p.predictions)
    probs = torch.sigmoid(logits).numpy()

    # Threshold for binary prediction (tune this later!)
    threshold = 0.3
    preds = (probs >= threshold).astype(int)

    labels = p.label_ids

    # Macro F1 (average across all labels)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)

    # Subset accuracy (exact match across all labels)
    accuracy = accuracy_score(labels, preds)

    # Per-label F1 for debugging
    f1_per_label = f1_score(labels, preds, average=None, zero_division=0)

    metrics = {
        "f1_macro": f1_macro,
        "accuracy": accuracy,
    }

    # Add per-label F1 scores
    for i, label in enumerate(label_cols):
        metrics[f"f1_{label}"] = f1_per_label[i]

    return metrics

In [10]:
MODEL_NAME = "xlm-roberta-large"  # Start with base, not large (faster)
MAX_LEN = 128

print(f"\nLoading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create datasets
print("Creating datasets...")
train_dataset = PolarizationTypeDataset(
    train["text"].tolist(),
    train[label_cols].values.tolist(),
    tokenizer,
    max_length=MAX_LEN,
)

val_dataset = PolarizationTypeDataset(
    validation["text"].tolist(),
    validation[label_cols].values.tolist(),
    tokenizer,
    max_length=MAX_LEN,
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")

# Build model
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
)

print(f"Model loaded. Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


Loading model: xlm-roberta-large


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Creating datasets...
Train dataset size: 13456
Val dataset size: 3365


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded. Device: cuda


In [11]:
output_dir = "./subtask2_baseline"

training_args = TrainingArguments(
    output_dir=output_dir,

    # Training hyperparameters (conservative starting point)
    num_train_epochs=8,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    # Regularization
    weight_decay=0.01,
    warmup_ratio=0.1,

    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,

    # Model selection
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    # Performance
    fp16=torch.cuda.is_available(),
    # fp16=False,  # DISABLED - CPU only, no mixed precision
    # use_cpu=True,  # Force CPU usage
    # no_cuda=True,

    # Reproducibility
    seed=GLOBAL_SEED,
    data_seed=GLOBAL_SEED,

    # Misc
    report_to="none",
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_multilabel,
    data_collator=DataCollatorWithPadding(tokenizer),
)

print("\nTrainer initialized. Ready to train!")
print(f"Training device: {trainer.args.device}")


Trainer initialized. Ready to train!
Training device: cuda:0


In [12]:
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60 + "\n")

trainer.train()


STARTING TRAINING



Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy,F1 Political,F1 Racial/ethnic,F1 Religious,F1 Gender/sexual,F1 Other
1,0.288700,0.278790,0.587367,0.516196,0.660764,0.605099,0.619718,0.645489,0.405762
2,0.249600,0.263492,0.591631,0.636256,0.675229,0.640288,0.587992,0.704731,0.349914
3,0.207700,0.249996,0.626720,0.591382,0.683434,0.668990,0.628664,0.690382,0.462131
4,0.168300,0.258308,0.638916,0.635661,0.694166,0.696897,0.632509,0.711238,0.459770
5,0.112500,0.309942,0.622137,0.637147,0.684369,0.674216,0.589319,0.720117,0.442667
6,0.105200,0.349891,0.626301,0.645765,0.664958,0.682158,0.600000,0.727838,0.456550
7,0.074700,0.370506,0.630983,0.645468,0.677882,0.684746,0.613475,0.720247,0.458564
8,0.049000,0.376612,0.631228,0.642496,0.684630,0.692180,0.613718,0.712782,0.452830


TrainOutput(global_step=6728, training_loss=0.16839476737695416, metrics={'train_runtime': 3141.9643, 'train_samples_per_second': 34.261, 'train_steps_per_second': 2.141, 'total_flos': 1.1749997098501536e+16, 'train_loss': 0.16839476737695416, 'epoch': 8.0})

In [17]:
print("\n" + "="*60)
print("FINAL EVALUATION")
print("="*60 + "\n")

final_metrics = trainer.evaluate()

trainer.save_model("./final_model/")
tokenizer.save_pretrained("./final_model/")

print("Validation Results:")
print(f"Macro F1: {final_metrics['eval_f1_macro']:.4f}")
print(f"Accuracy: {final_metrics['eval_accuracy']:.4f}")

print("\nPer-label F1 scores:")
for label in label_cols:
    score = final_metrics.get(f'eval_f1_{label}', 0)
    print(f"  {label}: {score:.4f}")

train_results = trainer.evaluate(train_dataset)
print(f"\nTrain - Loss: {train_results['eval_loss']:.4f}, Macro F1: {train_results['eval_f1_macro']:.4f}")

# Evaluate on validation set
val_results = trainer.evaluate(val_dataset)
print(f"Val   - Loss: {val_results['eval_loss']:.4f}, Macro F1: {val_results['eval_f1_macro']:.4f}")


FINAL EVALUATION



Validation Results:
Macro F1: 0.6389
Accuracy: 0.6357

Per-label F1 scores:
  political: 0.6942
  racial/ethnic: 0.6969
  religious: 0.6325
  gender/sexual: 0.7112
  other: 0.4598

Train - Loss: 0.1065, Macro F1: 0.8526
Val   - Loss: 0.2583, Macro F1: 0.6389
